# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [ ]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "PatchTST_OLCV"
REPO_DIR = "/content/ECE1508_GenAI"   # absolute path -- see note below

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    # git -C targets REPO_DIR explicitly rather than `cd X && ...`, so this is correct
    # regardless of the kernel's current working directory when the cell reruns.
    !git -C {REPO_DIR} pull

# Absolute path, not "ECE1508_GenAI": os.path.isdir("ECE1508_GenAI") above is checked
# relative to the CURRENT working directory -- on a second run of this cell (after the
# %cd below already moved the kernel into /content/ECE1508_GenAI), that relative check
# looks for /content/ECE1508_GenAI/ECE1508_GenAI, finds nothing, and silently clones a
# second, nested copy of the repo inside the first one instead of pulling it.
%cd {REPO_DIR}

In [3]:
# torch is preinstalled on Colab; transformers is needed for the HF PatchTSTModel-backed
# train_patchtst.py (src/models/patchtst_hf.py); mplfinance/pyyaml are for evaluate.py/configs.
!pip install -q transformers mplfinance pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.7 MB/s eta 0:00:00


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [ ]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

## Train PatchTST (HF PatchTSTModel-backed, real defaults: 20k windows/epoch, 20 epochs, configs/patchtst.yaml)

Now trains `src/models/patchtst_hf.py` instead of the original hand-rolled `src/models/patchtst.py`
-- see `docs/experiments.md` for the comparison. `channel_attention=False` is the config default
(the main approach for now); add `--channel-attention` below to try the mixing variant instead.
Saves to `steven/outputs/patchtst_hf_checkpoint.pt` (separate from the original model's checkpoint).

Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first
instead of the full config.

In [ ]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst.yaml --device auto

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae.yaml)

In [ ]:
!python steven/src/train_cvae.py --config steven/configs/cvae.yaml --device auto

## Evaluate both models on the fixed test set

**Not yet updated for the HF PatchTST model** -- `evaluate.py` still assumes the original
`patchtst.py`'s variable-context, padding-mask-aware interface, and the cell below still
points at `patchtst_checkpoint.pt` (the original model's checkpoint, not the new
`patchtst_hf_checkpoint.pt`). Only run this cell if you've separately trained the original
model too; otherwise skip it until evaluate.py is updated for the HF model.

In [5]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_false_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt

22:13:50 device: cuda
22:13:51 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
22:13:51 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
22:13:51 sell-price shrink bound: p99.0 of |anchored log return| over train = 0.0190 (vs. model's own MAX_LOG_RETURN)
22:13:51 patchtst checkpoint architecture: hf
22:13:51 evaluating on 2397 fixed test windows
22:13:52 wrote metrics to steven/outputs/metrics.json
22:13:52 overall: {
  "n_windows": 2397,
  "patchtst_reparam_mae_rmse": [
    0.12053213268518448,
    0.4461057782173157
  ],
  "cvae_reparam_mae_rmse": [
    0.14835265278816223,
    0.49762964248657227
  ],
  "patchtst_ohlc_mae_rmse": [
    3.5595765456527655,
    4.694126093176167
  ],
  "cvae_ohlc_mae_rmse": [
    3.0942732050571844,
    4.175779559808518
  ],
  "patchtst_volume_mae_rmse": [
    2313333.0,
    3876476.25
  ],
  "cvae_volume_mae_rmse": [
    2922454.25,
    4314411.0
  ],
  "patchtst_directional_accuracy": [
    0.4

In [ ]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_true_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt

## Refresh v1.md from this run

Rewrites the Results/backtest tables and sample images in `steven/v1.md` from the metrics.json + sample_plots this run just produced (see `steven/src/update_report.py`). Only the tables/images are rewritten -- surrounding prose (interpretation, caveats) is left as-is; review it by hand if the story changed. This only edits the file in the cloned repo here -- push/download separately if you want to keep it.

In [6]:
!python steven/src/update_report.py

22:16:49 updated steven/v1.md: results-samples, hit-summary, spread-summary, backtest-patchtst, backtest-cvae, buy-hold-benchmark
22:16:49 not auto-updated -- reread and edit by hand if the story changed: the 'In plain terms' / 'A subtle but important point' interpretation paragraphs under Results, the 'pre-retrain checkpoints' caveats in Results and Long-only backtest results, and the 'Retrain both models' checkbox under Next steps.


## Pull results back down

Zips `steven/outputs/` (checkpoints, metrics.json, sample_plots) and downloads it -- or just `git add`/`commit`/`push` from here if you'd rather sync back through the repo.

In [7]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")

  adding: steven/outputs/ (stored 0%)
  adding: steven/outputs/patchtst_false_checkpoint.pt (deflated 9%)
  adding: steven/outputs/metrics.json (deflated 86%)
  adding: steven/outputs/cvae_checkpoint.pt (deflated 9%)
  adding: steven/outputs/sample_plots/ (stored 0%)
  adding: steven/outputs/sample_plots/samples.json (deflated 77%)
  adding: steven/outputs/sample_plots/sample0_start24648_ctx70.png (deflated 11%)
  adding: steven/outputs/sample_plots/sample1_start25496_ctx70.png (deflated 10%)
  adding: steven/outputs/sample_plots/sample2_start26426_ctx70.png (deflated 11%)
  adding: steven/outputs/sample_plots/sample4_start26218_ctx70.png (deflated 10%)
  adding: steven/outputs/sample_plots/sample3_start24747_ctx70.png (deflated 10%)
  adding: steven/outputs/patchtst_true_checkpoint.pt (deflated 10%)
  adding: steven/outputs/patchtst_checkpoint.pt (deflated 10%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Or: commit results straight back to the repo (recommended)

`files.download()` above requires Colab's own hosted web frontend and won't trigger a
download when connected via a different client (e.g. Cursor's Colab GPU extension) --
if nothing downloaded from the cell above, use this instead. Commits
`steven/outputs/` (checkpoints, `metrics.json`, `sample_plots/`) and `steven/v1.md`
from this Colab session directly to `origin/{BRANCH}`; pull locally afterward to sync.

In [ ]:
!git add steven/outputs steven/v1.md
!git status
!git commit -m "chore(model): update metrics.json/sample_plots/v1.md from this Colab run"
!git push origin {BRANCH}